<div align="center">

# Predicción de tasas de mortalidad y natalidad a partir de indicadores socioeconómicos

**Proyecto Integrador**

Universidad Internacional del Ecuador — UIDE

---

**Notebook 03 — Modelado y Evaluación**

---

Equipo: Guillermo Paredes · Donato Oña · Mateo Villacreses

Junio 2026

</div>

## Objetivo

Entrenar y comparar un conjunto de modelos de regresión supervisada para predecir las tasas brutas de mortalidad y natalidad, empleando el dataset procesado en el notebook anterior. El análisis aborda cinco preguntas:

1. **¿Cuál es el desempeño de una línea base lineal frente a modelos no lineales?** El EDA sugirió que la mortalidad presenta una relación no monótona con el desarrollo (paradoja demográfica en forma de U invertida). Si esa hipótesis es correcta, los modelos basados en árboles deberían superar a los lineales con un margen mayor para `death_rate` que para `birth_rate`.
2. **¿Qué variables resultan más informativas para cada target?** Se cuantifica mediante importancia por permutación.
3. **¿Cuánto del desempeño en `birth_rate` proviene del feature de estructura etaria `pop_0to14_pct`?** Este indicador presenta una correlación muy alta con la tasa de natalidad, lo cual sugiere una identidad demográfica más que una relación causal.
4. **¿Generaliza el modelo a países no observados durante el entrenamiento?** La evaluación se realiza mediante `GroupKFold` agrupado por código de país.
5. **¿Los resultados son robustos al choque 2020-2021?** Se compara el desempeño usando todos los años frente a una evaluación sin años pandémicos.

## Estructura del notebook

1. Configuración inicial y carga del dataset procesado.
2. Definición explícita de predictores y targets.
3. Esquema de validación cruzada por país.
4. Definición de los modelos a comparar.
5. Entrenamiento y evaluación.
6. Visualización: predicciones vs. valores reales.
7. Importancia de features por permutación.
8. Estudio del aporte real de `pop_0to14_pct`.
9. Análisis de residuales.
10. Robustez frente a años de pandemia.
11. Discusión y conclusiones.


## 1. Configuración inicial

In [ ]:
# Configuración para Google Colab
import sys
if 'google.colab' in sys.modules:
    print("Ejecutando en Google Colab. Instalando dependencias necesarias...")
    !pip install -q wbgapi pyarrow xgboost lightgbm shap


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_validate, cross_val_predict
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore', category=UserWarning)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Carga del dataset procesado

Se cargan los tres artefactos producidos por el notebook 02:

- `dataset_modelado.parquet`: dataset procesado completo, usado como contrato canónico del pipeline.
- `train_processed.parquet`: países reservados para entrenamiento y validación cruzada.
- `test_processed.parquet`: países nunca usados para entrenar, útil para importancia por permutación y chequeos fuera de muestra.


In [ ]:
# Cargar los datasets procesados desde el notebook 02
import os
import pandas as pd

processed_path = '../data/processed/dataset_modelado.parquet'
train_path = '../data/processed/train_processed.parquet'
test_path = '../data/processed/test_processed.parquet'

missing_paths = [p for p in [processed_path, train_path, test_path] if not os.path.exists(p)]
if missing_paths:
    raise FileNotFoundError(
        "ERROR: faltan artefactos procesados. Ejecuta primero 02_preprocesamiento.ipynb. "
        f"No encontrados: {missing_paths}"
    )

df_all = pd.read_parquet(processed_path)
df_train = pd.read_parquet(train_path)
df_test = pd.read_parquet(test_path)

assert set(df_train['country_code']).isdisjoint(set(df_test['country_code'])), "Train/Test comparten países"

# El modelado y la comparación de modelos mediante CV se realizan estrictamente sobre entrenamiento.
df = df_train.copy()

print("Datasets cargados exitosamente.")
print(f"  Completo: {df_all.shape} | países: {df_all['country_code'].nunique()}")
print(f"  Train:    {df_train.shape} | países: {df_train['country_code'].nunique()}")
print(f"  Test:     {df_test.shape} | países: {df_test['country_code'].nunique()}")


## 3. Definición de variables, predictores y targets

Se separa explícitamente el conjunto de columnas en tres roles:

- **Metadatos** (`country_code`, `country_name`, `region`, `income_level`, `year`): no se emplean para el entrenamiento pero se conservan para análisis de errores e interpretación.
- **Variables objetivo** (`death_rate`, `birth_rate`): cada una se modela de forma independiente.
- **Features del modelo**: una lista cerrada y explicable. Se usan variables económicas, sociales, estructura etaria, tiempo, pandemia, nivel de ingreso, región y la bandera de imputación del Gini. Las columnas originales muy sesgadas (`gdp_per_capita`, `population`) se conservan en el dataset, pero el modelo usa sus versiones logarítmicas.


In [ ]:
META_COLS = ['country_code', 'country_name', 'region', 'income_level', 'year']
TARGETS = ['death_rate', 'birth_rate']

# Features explícitos: conservamos las columnas originales para trazabilidad,
# pero el modelo usa las versiones logarítmicas de PIB per cápita y población.
BASE_FEATURES = [
    'gdp_growth',
    'gini',
    'unemployment',
    'health_exp_pct_gdp',
    'edu_exp_pct_gdp',
    'urban_pop_pct',
    'pop_65plus_pct',
    'pop_0to14_pct',
    'log_gdp_per_capita',
    'log_population',
    'year_norm',
    'is_pandemic',
    'income_level_ord',
    'gini_imputed',
]
REGION_FEATURES = sorted(c for c in df.columns if c.startswith('region_'))
FEATURES = BASE_FEATURES + REGION_FEATURES

missing_features = [c for c in FEATURES if c not in df.columns]
assert not missing_features, f"Faltan features esperados: {missing_features}"
assert 'region_' not in FEATURES, "Apareció region_ vacío: revisar filtro de agregados del Banco Mundial"
assert df[FEATURES].isna().sum().sum() == 0, "Hay NaN en features de modelado"

print(f"Targets ({len(TARGETS)}): {TARGETS}")
print(f"Features usadas por el modelo ({len(FEATURES)}):")
for c in FEATURES:
    print(f"  - {c}")


In [ ]:
X = df[FEATURES].values
y_death = df['death_rate'].values
y_birth = df['birth_rate'].values
groups = df['country_code'].values

print(f"X shape:        {X.shape}")
print(f"y_death shape:  {y_death.shape}  (rango: {y_death.min():.2f}–{y_death.max():.2f})")
print(f"y_birth shape:  {y_birth.shape}  (rango: {y_birth.min():.2f}–{y_birth.max():.2f})")
print(f"Grupos únicos:  {len(np.unique(groups))}  (países distintos)")

## 4. Esquema de validación cruzada

Se emplea `GroupKFold` con 5 pliegues, agrupando por código de país: ningún país aparece simultáneamente en el conjunto de entrenamiento y en el de validación de un mismo pliegue. Este esquema produce estimaciones de generalización a **países no observados** durante el entrenamiento, lo cual responde al objetivo del proyecto.

In [ ]:
N_SPLITS = 5
cv = GroupKFold(n_splits=N_SPLITS)

# Verificar tamaños de los pliegues
print(f"Esquema: GroupKFold con {N_SPLITS} pliegues\n")
for i, (train_idx, val_idx) in enumerate(cv.split(X, y_death, groups), 1):
    n_train = len(train_idx)
    n_val = len(val_idx)
    n_train_countries = len(np.unique(groups[train_idx]))
    n_val_countries = len(np.unique(groups[val_idx]))
    print(f"  Pliegue {i}: train={n_train} filas / {n_train_countries} países  |  val={n_val} filas / {n_val_countries} países")

## 5. Modelos a comparar

Se evalúan cuatro modelos cubriendo el espectro de complejidad habitual en problemas tabulares:

| Modelo | Familia | Función |
|---|---|---|
| **Ridge** | Lineal regularizado | Línea base; cuantifica la fracción de variabilidad explicable por relaciones lineales tras normalización. |
| **Random Forest** | Ensamble de árboles (bagging) | Captura relaciones no lineales y heterogéneas sin necesidad de tuning agresivo. |
| **XGBoost** | Ensamble de árboles (boosting) | Suele ofrecer el mejor desempeño en problemas tabulares con tamaño moderado. |
| **LightGBM** | Ensamble de árboles (boosting) | Alternativa eficiente a XGBoost; útil para verificar consistencia entre implementaciones. |

Los hiperparámetros se fijan a valores razonables sin optimización; el objetivo de este notebook es establecer una línea base de comparación entre familias de modelos. La búsqueda de hiperparámetros se aborda en un notebook posterior, en caso de que el desempeño actual lo justifique.

In [ ]:
def construir_modelos(random_state=RANDOM_STATE):
    return {
        'Ridge': Pipeline([
            ('scaler', StandardScaler()),
            ('reg', Ridge(alpha=1.0, random_state=random_state)),
        ]),
        'RandomForest': RandomForestRegressor(
            n_estimators=300, max_depth=None, min_samples_leaf=2,
            random_state=random_state, n_jobs=-1,
        ),
        'XGBoost': xgb.XGBRegressor(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            random_state=random_state, n_jobs=-1, verbosity=0,
        ),
        'LightGBM': lgb.LGBMRegressor(
            n_estimators=400, max_depth=-1, num_leaves=31, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            random_state=random_state, n_jobs=-1, verbose=-1,
        ),
    }

modelos_de_referencia = construir_modelos()
print("Modelos definidos:", list(modelos_de_referencia.keys()))

## 6. Entrenamiento y evaluación con validación cruzada

Cada combinación (modelo × target) se evalúa mediante validación cruzada agrupada por país. Las métricas reportadas son:

- **RMSE** (Root Mean Squared Error) — métrica principal; expresada en las mismas unidades que el target (tasas por 1.000 habitantes).
- **MAE** (Mean Absolute Error) — robusta a outliers; complementa al RMSE.
- **R²** — fracción de varianza explicada; útil como métrica adimensional.

In [ ]:
scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2',
}

targets_y = {'death_rate': y_death, 'birth_rate': y_birth}
registros = []

for target_name, y in targets_y.items():
    for nombre_modelo, modelo in construir_modelos().items():
        scores = cross_validate(
            modelo, X, y, groups=groups, cv=cv,
            scoring=scoring, n_jobs=-1, return_train_score=False,
        )
        registros.append({
            'target': target_name,
            'modelo': nombre_modelo,
            'RMSE_mean': -scores['test_rmse'].mean(),
            'RMSE_std':   scores['test_rmse'].std(),
            'MAE_mean':  -scores['test_mae'].mean(),
            'R2_mean':    scores['test_r2'].mean(),
            'R2_std':     scores['test_r2'].std(),
            'fit_time':   scores['fit_time'].mean(),
        })
        print(f"  ✔ {target_name:11s} × {nombre_modelo:12s} "
              f"RMSE={-scores['test_rmse'].mean():.3f}  R²={scores['test_r2'].mean():.3f}")

resultados = pd.DataFrame(registros)
print("\n✅ Validación cruzada completada.")

In [ ]:
# Tabla ordenada por target y RMSE
tabla = resultados.copy()
tabla['RMSE'] = tabla.apply(lambda r: f"{r['RMSE_mean']:.3f} ± {r['RMSE_std']:.3f}", axis=1)
tabla['R²'] = tabla.apply(lambda r: f"{r['R2_mean']:.3f} ± {r['R2_std']:.3f}", axis=1)
tabla['MAE'] = tabla['MAE_mean'].round(3)
tabla = tabla[['target', 'modelo', 'RMSE', 'MAE', 'R²']].sort_values(['target', 'modelo']).reset_index(drop=True)
tabla

In [ ]:
# Visualización de la comparación
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, target in zip(axes, TARGETS):
    sub = resultados[resultados['target'] == target].sort_values('RMSE_mean')
    bars = ax.barh(sub['modelo'], sub['RMSE_mean'], xerr=sub['RMSE_std'],
                   color=['#2c7fb8', '#7fcdbb', '#fdae61', '#d7191c'], capsize=4)
    ax.set_xlabel('RMSE (validación cruzada)')
    ax.set_title(f'{target} — comparación de modelos')
    ax.grid(axis='x', alpha=0.3)
    # Anotar R² al lado
    for bar, r2 in zip(bars, sub['R2_mean']):
        ax.text(bar.get_width() * 1.02, bar.get_y() + bar.get_height()/2,
                f'R²={r2:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Predicciones vs valores reales

Se obtienen las predicciones fuera de muestra usando `cross_val_predict` —de forma que cada observación se predice cuando su país no formó parte del entrenamiento— y se contrastan con los valores reales. La cercanía a la diagonal `y = ŷ` indica buen desempeño.

In [ ]:
# Identificar el mejor modelo por target (menor RMSE)
mejores = resultados.loc[resultados.groupby('target')['RMSE_mean'].idxmin()]
print("Mejor modelo por target:")
print(mejores[['target', 'modelo', 'RMSE_mean', 'R2_mean']].to_string(index=False))

In [ ]:
# Predicciones fuera de muestra con el mejor modelo de cada target
predicciones_oof = {}

for _, row in mejores.iterrows():
    target_name = row['target']
    modelo_nombre = row['modelo']
    modelo = construir_modelos()[modelo_nombre]
    y = targets_y[target_name]
    y_pred = cross_val_predict(modelo, X, y, groups=groups, cv=cv, n_jobs=-1)
    predicciones_oof[target_name] = (modelo_nombre, y, y_pred)
    print(f"  ✔ {target_name} ({modelo_nombre}): RMSE={np.sqrt(mean_squared_error(y, y_pred)):.3f}, R²={r2_score(y, y_pred):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, (target, (modelo_nombre, y_true, y_pred)) in zip(axes, predicciones_oof.items()):
    # Color por región
    region_codes = df['region'].values
    palette = sns.color_palette('tab10', n_colors=len(np.unique(region_codes)))
    region_to_color = dict(zip(np.unique(region_codes), palette))
    colors = [region_to_color[r] for r in region_codes]

    ax.scatter(y_true, y_pred, c=colors, alpha=0.4, s=14, edgecolor='none')
    lim_min = min(y_true.min(), y_pred.min())
    lim_max = max(y_true.max(), y_pred.max())
    ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', lw=1, label='y = ŷ')
    ax.set_xlabel(f'{target} real')
    ax.set_ylabel(f'{target} predicho')
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    ax.set_title(f'{target} — {modelo_nombre}\nRMSE = {rmse:.3f}  |  R² = {r2:.3f}')
    ax.legend(loc='upper left')

# Leyenda de colores
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, label=r, markersize=7)
           for r, c in region_to_color.items()]
fig.legend(handles=handles, loc='lower center', ncol=len(handles), bbox_to_anchor=(0.5, -0.05), fontsize=9)
plt.tight_layout()
plt.show()

## 8. Importancia de features por permutación

Para el mejor modelo de cada target, se cuantifica la importancia de cada feature mediante **permutation importance**: la métrica de evaluación se recalcula varias veces tras permutar aleatoriamente los valores de cada feature; cuanto mayor la caída del desempeño, mayor la importancia del feature.

Esta técnica es **agnóstica al modelo** y mide importancia *predictiva* (no causal), lo cual es preferible a los valores `feature_importances_` nativos de los modelos de árboles —que pueden sobrerrepresentar a features con alta cardinalidad.

In [ ]:
# Evaluamos importancia en el conjunto de prueba (Test) real obtenido de 02_preprocesamiento.ipynb
X_train = df_train[FEATURES].values
X_test = df_test[FEATURES].values

print(f"Train: {X_train.shape[0]} filas / {df_train['country_code'].nunique()} países")
print(f"Test:  {X_test.shape[0]} filas / {df_test['country_code'].nunique()} países")


In [ ]:
importancias = {}

for _, row in mejores.iterrows():
    target_name = row['target']
    modelo_nombre = row['modelo']
    modelo = construir_modelos()[modelo_nombre]
    
    y_train = df_train[target_name].values
    y_test = df_test[target_name].values

    modelo.fit(X_train, y_train)
    pi = permutation_importance(
        modelo, X_test, y_test,
        n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1, scoring='neg_root_mean_squared_error',
    )
    importancias[target_name] = pd.DataFrame({
        'feature': FEATURES,
        'mean': pi.importances_mean,
        'std': pi.importances_std,
    }).sort_values('mean', ascending=False).reset_index(drop=True)
    print(f"  ✔ Importancia calculada para {target_name} ({modelo_nombre})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (target, imp_df) in zip(axes, importancias.items()):
    top = imp_df.head(12).iloc[::-1]  # invertir para barh de abajo a arriba
    ax.barh(top['feature'], top['mean'], xerr=top['std'], color='#2c7fb8', capsize=3)
    ax.set_xlabel('Pérdida de RMSE al permutar (mayor = más importante)')
    ax.set_title(f'Importancia de features — {target}')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

for target, imp_df in importancias.items():
    print(f"\nTop 5 features para {target}:")
    print(imp_df.head(5).to_string(index=False))

## 9. Estudio del aporte real de `pop_0to14_pct`

La correlación de Spearman entre `pop_0to14_pct` y `birth_rate` es de 0.977, valor que sugiere una **identidad demográfica** más que una relación causal: el porcentaje de población menor de 14 años refleja la acumulación de la natalidad reciente. Para cuantificar el aporte real de los **predictores socioeconómicos puros** se entrena un modelo análogo excluyendo las variables de estructura etaria.

In [ ]:
FEATURES_SIN_EDAD = [c for c in FEATURES if c not in ['pop_0to14_pct', 'pop_65plus_pct']]
X_sin_edad = df[FEATURES_SIN_EDAD].values

print(f"Features sin estructura etaria: {len(FEATURES_SIN_EDAD)} (vs {len(FEATURES)} originales)")

comparacion_edad = []
for target_name, y in targets_y.items():
    for set_name, X_set in [('con_edad', X), ('sin_edad', X_sin_edad)]:
        modelo = construir_modelos()['XGBoost']  # usamos el mismo modelo para comparar justo
        scores = cross_validate(
            modelo, X_set, y, groups=groups, cv=cv,
            scoring=scoring, n_jobs=-1,
        )
        comparacion_edad.append({
            'target': target_name,
            'features': set_name,
            'RMSE': -scores['test_rmse'].mean(),
            'R²': scores['test_r2'].mean(),
        })

df_comp_edad = pd.DataFrame(comparacion_edad)
print("\nComparación con vs sin features de estructura etaria (modelo XGBoost):\n")
print(df_comp_edad.round(3).to_string(index=False))

Esta comparación cuantifica explícitamente qué fracción del desempeño proviene de la estructura etaria y qué fracción de los predictores propiamente socioeconómicos. La interpretación correcta es:

- El modelo **con estructura etaria** responde a la pregunta: *dados todos los indicadores demográficos y socioeconómicos disponibles, ¿qué tan precisa puede ser la predicción?*
- El modelo **sin estructura etaria** responde a la pregunta: *¿qué tanta señal sobre las tasas demográficas puede recuperarse a partir únicamente de variables socioeconómicas?*

Ambas preguntas son legítimas y la comparación entre ambas es uno de los hallazgos centrales del proyecto.

## 10. Análisis de residuales

Se examina la distribución de los residuales del mejor modelo para cada target. Idealmente, los residuales deberían:

1. Estar centrados en cero (sin sesgo sistemático).
2. Tener varianza aproximadamente constante en todo el rango de predicciones (homocedasticidad).
3. No presentar patrones temporales ni geográficos sistemáticos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for col, (target, (modelo_nombre, y_true, y_pred)) in enumerate(predicciones_oof.items()):
    residuales = y_true - y_pred

    # Histograma de residuales
    axes[0, col].hist(residuales, bins=50, color='steelblue', edgecolor='white')
    axes[0, col].axvline(0, color='red', linestyle='--', alpha=0.7)
    axes[0, col].set_xlabel(f'Residual ({target})')
    axes[0, col].set_ylabel('Frecuencia')
    axes[0, col].set_title(
        f'{target} — distribución de residuales\n'
        f'media = {residuales.mean():+.3f}, desv.est = {residuales.std():.3f}'
    )

    # Residuales vs predicción
    axes[1, col].scatter(y_pred, residuales, alpha=0.3, s=10, color='steelblue')
    axes[1, col].axhline(0, color='red', linestyle='--', alpha=0.7)
    axes[1, col].set_xlabel(f'{target} predicho')
    axes[1, col].set_ylabel('Residual')
    axes[1, col].set_title(f'{target} — residuales vs predicción')

plt.tight_layout()
plt.show()

### Casos con mayor error

Examinar los países con los residuales más grandes ayuda a identificar limitaciones del modelo: si hay patrones (por ejemplo, países pequeños, países en conflicto, microestados), puede orientar mejoras futuras.

In [ ]:
for target, (modelo_nombre, y_true, y_pred) in predicciones_oof.items():
    residuales = y_true - y_pred
    df_err = df[['country_code', 'country_name', 'region', 'year']].copy()
    df_err['real'] = y_true
    df_err['predicho'] = y_pred
    df_err['residual'] = residuales
    df_err['abs_residual'] = np.abs(residuales)

    top_err = df_err.nlargest(10, 'abs_residual')[
        ['country_name', 'region', 'year', 'real', 'predicho', 'residual']
    ].round(2)
    print(f"\nTop 10 errores absolutos para {target} ({modelo_nombre}):")
    print(top_err.to_string(index=False))

## 11. Robustez frente a años de pandemia

Se compara el desempeño de XGBoost usando todos los años disponibles frente a una evaluación que excluye 2020-2021. La idea no es modelar causalmente la pandemia, sino comprobar si las conclusiones principales dependen demasiado de ese choque global.


In [ ]:
robustez_pandemia = []

for target_name in TARGETS:
    for escenario, data_eval in [
        ('todos_los_anios', df),
        ('sin_2020_2021', df[df['is_pandemic'] == 0]),
    ]:
        X_eval = data_eval[FEATURES].values
        y_eval = data_eval[target_name].values
        groups_eval = data_eval['country_code'].values
        scores = cross_validate(
            construir_modelos()['XGBoost'],
            X_eval,
            y_eval,
            groups=groups_eval,
            cv=GroupKFold(n_splits=N_SPLITS),
            scoring=scoring,
            n_jobs=-1,
        )
        robustez_pandemia.append({
            'target': target_name,
            'escenario': escenario,
            'RMSE': -scores['test_rmse'].mean(),
            'MAE': -scores['test_mae'].mean(),
            'R2': scores['test_r2'].mean(),
        })

df_robustez_pandemia = pd.DataFrame(robustez_pandemia)
print("Robustez frente a años pandémicos (XGBoost, GroupKFold):\n")
print(df_robustez_pandemia.round(3).to_string(index=False))


## 12. Discusión y conclusiones

### Hallazgos principales

1. **Desempeño general.** Las celdas anteriores seleccionan automáticamente el mejor modelo por menor RMSE. En las ejecuciones previas del proyecto, XGBoost fue el mejor compromiso para ambos targets: natalidad se predice con R² alto, mientras que mortalidad mantiene un techo más bajo.

2. **Lectura sencilla del modelo.** El modelo usa cuatro bloques fáciles de explicar: desarrollo económico, gasto/condiciones sociales, estructura etaria y contexto país-año (región, ingreso, año, pandemia). Esto permite defender el pipeline sin presentar al modelo como una caja negra arbitraria.

3. **Mortalidad es más difícil que natalidad.** La mortalidad cruda no depende solo del desarrollo: también refleja envejecimiento, crisis sanitarias, conflictos, epidemias y calidad del registro. Por eso los modelos no lineales ayudan, pero no eliminan los errores extremos.

4. **Natalidad tiene una identidad demográfica.** `pop_0to14_pct` es muy predictiva porque resume la acumulación reciente de nacimientos. La comparación con y sin estructura etaria permite comunicar una métrica honesta de señal socioeconómica pura.

5. **Importancias y residuales.** La importancia por permutación se interpreta como utilidad predictiva, no causalidad. Los residuales grandes se revisan por país, región y año para detectar contextos que el set de variables no captura bien.

6. **Pandemia.** `is_pandemic` identifica 2020-2021 como choque global. La prueba de robustez muestra si las conclusiones cambian al excluir esos años.

### Limitaciones

- **Gini imputado.** El coeficiente de Gini tiene baja cobertura y mide solo desigualdad de ingresos; su importancia debe leerse junto a `gini_imputed`.
- **Ground truth modelado.** Las tasas del Banco Mundial son estimaciones demográficas, no registros civiles crudos homogéneos.
- **Variables ausentes.** Faltan gobernanza, conflictos, capacidad sanitaria, vacunación, violencia y calidad institucional.
- **Generalización ≠ proyección.** `GroupKFold` por país evalúa países nuevos, no predicción del futuro. Una tarea de pronóstico requeriría partición temporal.

### Siguientes pasos sugeridos

1. Optimizar hiperparámetros del modelo ganador, especialmente para `death_rate`.
2. Usar SHAP para interpretar efectos globales y locales por región/nivel de ingreso.
3. Incorporar variables institucionales y sanitarias para atacar los residuales extremos.
4. Evaluar una partición temporal si el objetivo cambia hacia proyección demográfica.
